In [23]:
import pdfplumber
from pathlib import Path
from pdfplumber.utils import extract_text
import re
from random import choice
import shutil

In [4]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [ ]:
# file = ROOT / Path("examenes_quimica/2026_quimica/2026 - quimica - ordinaria titular titular quínica.pdf")

In [ ]:
# with open("primeras_lineas.txt", "w") as f:
#     for folder in sorted((ROOT / "examenes_quimica").iterdir()):
#         if folder.is_dir():
#             chosen_pdf = Path(choice(list(folder.iterdir())))
#             f.write("\t" + " - ".join(chosen_pdf.parts[-2:]) + "\n")
#             with pdfplumber.open(chosen_pdf) as pdf:
#                 for p in pdf.pages:
#                     lines = "\n".join(p.extract_text().split("\n")[:5]) + "\n\n"
#                     f.write(lines)

In [7]:
def left_neighbor(char, chars_by_x1):
    """Carácter que termina justo donde empieza `char` (a menos de 1,5 pt) y a su misma altura."""
    x = round(char["x0"])
    candidates = [other for key in (x - 1, x, x + 1) for other in chars_by_x1.get(key, [])
                  if other is not char
                  and abs(other["x1"] - char["x0"]) < 1.5
                  and other["top"] < char["bottom"] and char["top"] < other["bottom"]]
    return min(candidates, key=lambda other: abs(other["x1"] - char["x0"]), default=None)

def extract_page_text(page):
    """Texto de la página convirtiendo los sub/superíndices en caracteres Unicode."""
    # índice de los caracteres por su borde derecho (redondeado) para encontrar vecinos rápido
    chars_by_x1 = {}
    for char in page.chars:
        chars_by_x1.setdefault(round(char["x1"]), []).append(char)

    base_of = {}       # id(índice) -> carácter normal al que acompaña (la "H" de "H₂")
    characters = []
    for char in sorted(page.chars, key=lambda char: char["x0"]):   # de izquierda a derecha
        left = left_neighbor(char, chars_by_x1)
        # si el vecino ya es un índice (el "¹" de "10⁻¹⁴"), la base es la de ese vecino
        base = base_of.get(id(left), left)
        if base is not None and char["size"] < 0.9 * base["size"]:
            # superíndice si su base queda claramente por encima de la del carácter normal
            table = SUPERSCRIPT if char["bottom"] < base["bottom"] - 2 else SUBSCRIPT
            base_of[id(char)] = base
            # se le da la altura de su base para que extract_text lo ponga en la misma línea
            char = dict(char, text=char["text"].translate(table), top=base["top"], bottom=base["bottom"])
        characters.append(char)
    # extract_text junta en una línea los caracteres cuyo top difiere como máximo y_tolerance
    # y dentro de cada línea los ordena por x0
    return extract_text(characters, y_tolerance=4)

In [9]:
def extract_exercises_from_pdf(file: Path) -> str:
    match_year = re.search(r"\d{4}", file.stem)
    if match_year:
        year = int(match_year.group(0))

    exam_text = f"{file.stem}\n\n"

    if year <= 2019:
        start_string = "OPCIÓN"
    else:
        start_string = r"BLOQUE|B\d{1}\."

    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            page_text = extract_page_text(page)
            
            match = re.search(start_string, page_text)
            if match:
                page_text = page_text[match.start():]
                page_text = (page_text.replace("OPCIÓN A", "Opción A\n")
                             .replace("OPCIÓN B", "Opción B\n")
                             )

            exam_text += f"{page_text}\n\n"

    return exam_text

Diccionarios de sustitución de caracteres con `str.maketrans()`

In [10]:
SUBSCRIPT = str.maketrans("0123456789+-–‒−", "₀₁₂₃₄₅₆₇₈₉₊₋₋₋₋")
SUPERSCRIPT = str.maketrans("0123456789+-–‒−", "⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁻⁻⁻")

Guardar los enunciados de cada examen en un archivo .txt

In [ ]:
for file  in sorted((ROOT / "examenes_quimica").rglob("*.pdf")):
    filename = str(file.resolve()).replace("pdf", "") + "txt"
    # remove .DS_Store files
    if file.name.startswith(".DS_Store"):
        file.unlink()
        continue

    # extract pdf text and save into text file
    with open(filename, "w") as f:
        try:
            f.write(extract_exercises_from_pdf(file))
        except Exception as e:
            print(f"Error: {file.name}")


In [ ]:
file = ROOT / "examenes_quimica/2022_quimica/2017 - quimica - titular septiembre 6.pdf"
file2 = ROOT / "examenes_quimica/2022_quimica/2022 - quimica - impreso ord titular .pdf"

In [ ]:
sept_2017_exercises = extract_exercises_from_pdf(file)
junio_2022_exercises = extract_exercises_from_pdf(file2)

In [ ]:
# print(sept_2017_exercises)

In [34]:
# print(junio_2022_exercises)